# Testing HMM AR(1) implementation:

In [4]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from methods.hmm_ar1 import (
    simulate_rs_ar1,
    transform_params,
    obs_density,
    forward_algorithm,
    neg_loglik,
    fit_model,
    filtered_probs
)

import numpy as np

### simulate_rs_ar1

In [5]:
T = 100
beta = np.array([0.2, 0.8])
sigma = np.array([0.5, 1.0])
P = np.array([[0.95, 0.05],
              [0.05, 0.95]])

y, states = simulate_rs_ar1(T, beta, sigma, P, seed=1)

print("y shape:", y.shape)
print("states shape:", states.shape)
print("unique states:", np.unique(states))
print("first 5 y:", y[:5])
print("first 5 states:", states[:5])

y shape: (100,)
states shape: (100,)
unique states: [0 1]
first 5 y: [-0.54974618 -1.84266966 -1.21654221 -2.93262836 -1.48792383]
first 5 states: [1 1 1 1 1]


In [6]:
assert len(y) == T
assert len(states) == T
assert set(np.unique(states)).issubset({0, 1})

### transform_params

In [7]:
theta = np.array([0.0, 1.0, -0.5, 0.2, 2.0, -1.0])

beta1, beta2, sigma1, sigma2, p11, p22 = transform_params(theta)

print(beta1, beta2, sigma1, sigma2, p11, p22)

0.0 0.46211715726000974 0.6065306597126334 1.2214027581601699 0.8807970779778823 0.2689414213699951


In [8]:
assert -1 < beta1 < 1
assert -1 < beta2 < 1
assert sigma1 > 0
assert sigma2 > 0
assert 0 < p11 < 1
assert 0 < p22 < 1

### obs_density

In [9]:
val = obs_density(y_t=0.0, y_tm1=0.0, beta=0.5, sigma=1.0)
print(val)

0.3989422804014327


In [10]:
assert np.isfinite(val)
assert val > 0

In [11]:
expected = 1 / np.sqrt(2 * np.pi)
print("difference:", abs(val - expected))

difference: 0.0


### forward_algorithm

In [12]:
alpha, c, loglik = forward_algorithm(
    y=y,
    beta1=0.2,
    beta2=0.8,
    sigma1=0.5,
    sigma2=1.0,
    p11=0.95,
    p22=0.95
)

print("alpha shape:", alpha.shape)
print("c shape:", c.shape)
print("loglik:", loglik)
print("row sums of alpha (first 5):", alpha.sum(axis=1)[:5])

alpha shape: (100, 2)
c shape: (100,)
loglik: -80.67898899875375
row sums of alpha (first 5): [1. 1. 1. 1. 1.]


In [13]:
assert alpha.shape == (T, 2)
assert c.shape == (T,)
assert np.isfinite(loglik)
assert np.allclose(alpha.sum(axis=1), 1.0)
assert np.all(c > 0)

### neg_loglik

In [14]:
theta0 = np.array([0.1, 0.1, 0.0, 0.0, 2.0, 2.0])
nll = neg_loglik(theta0, y)

print("negative loglik:", nll)

negative loglik: 112.02549144712506


In [15]:
assert np.isfinite(nll)
assert isinstance(nll, float) or np.isscalar(nll)

### fit_model

In [16]:
theta0 = np.array([0.1, 0.1, 0.0, 0.0, 2.0, 2.0])

result, params_hat = fit_model(y, theta0)

print("success:", result.success)
print("message:", result.message)
print("estimated params:", params_hat)

success: True
message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
estimated params: {'beta1': 0.45767586008887257, 'beta2': 0.45767586008887257, 'sigma1': 0.5771355897905505, 'sigma2': 0.5771355897905505, 'p11': 0.8807970998186221, 'p22': 0.8807970998186221, 'p12': 0.11920290018137791, 'p21': 0.11920290018137791}


In [17]:
assert result.x.shape == (6,)
assert all(k in params_hat for k in ["beta1", "beta2", "sigma1", "sigma2", "p11", "p22"])
assert params_hat["sigma1"] > 0
assert params_hat["sigma2"] > 0
assert 0 < params_hat["p11"] < 1
assert 0 < params_hat["p22"] < 1

### (filtered_probs)